# ED Phase-2 — Trainer (from scratch) + Isotonic Calibration\n\n**Purpose:** Train a small tabular model on your attached CSVs (`train_DE_full.csv`, `val_DE_full.csv`, `test_DE_full.csv`), calibrate probabilities with **isotonic regression**, enforce a **recall ≥ 0.85** floor on validation for the classification threshold, evaluate on test, and **save a scorer bundle** for your OPS notebook.\n\n**Outputs**\n- `ed_phase2_model.joblib` — pipeline + isotonic calibrator + threshold + metadata\n- `test_predictions.csv` — per-row calibrated probabilities on the test set\n- Metrics printed to stdout\n\n> This notebook is self-contained and does not train inside the OPS UI notebook.

In [ ]:

# --- Bootstrap: environment + paths + logger ---
import os, json, datetime as _dt
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else ("/content" if os.path.exists("/content") else "/mnt/data")
_p = lambda *p: os.path.join(BASE, *p)

CONFIG = {
    "TRAIN_PATH": _p("train_DE_full.csv"),
    "VAL_PATH":   _p("val_DE_full.csv"),
    "TEST_PATH":  _p("test_DE_full.csv"),
    "MODEL_OUT":  _p("ed_phase2_model.joblib"),
    "PRED_OUT":   _p("test_predictions.csv"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
}
os.makedirs(BASE, exist_ok=True)
def _append_event(ev: dict):
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")
print("BASE =", BASE)
print("Outputs →", CONFIG["MODEL_OUT"], "and", CONFIG["PRED_OUT"])


In [ ]:

# --- Resolve dataset files (Kaggle input or local) ---
import glob, shutil, os, pandas as pd

def _resolve(name):
    # Prefer pre-set CONFIG path if file exists
    if os.path.exists(CONFIG[name]):
        return CONFIG[name]
    # Try Kaggle /kaggle/input recursively
    for root in ["/kaggle/input", "/mnt/data", "/content"]:
        hits = glob.glob(os.path.join(root, "**", os.path.basename(CONFIG[name])), recursive=True)
        if hits:
            # copy to BASE for a clean path
            dst = CONFIG[name]
            if not os.path.exists(dst):
                try:
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(hits[0], dst)
                except Exception:
                    dst = hits[0]
            return dst
    raise FileNotFoundError(f"Could not locate {CONFIG[name]} in /kaggle/input or local paths. Attach/upload the CSV.")

CONFIG["TRAIN_PATH"] = _resolve("TRAIN_PATH")
CONFIG["VAL_PATH"]   = _resolve("VAL_PATH")
CONFIG["TEST_PATH"]  = _resolve("TEST_PATH")
print("Resolved:")
for k in ["TRAIN_PATH","VAL_PATH","TEST_PATH"]:
    print(f" - {k}: {CONFIG[k]}")

# Quick peek
for k in ["TRAIN_PATH","VAL_PATH","TEST_PATH"]:
    df = pd.read_csv(CONFIG[k], nrows=3)
    print(f"{k} head:")
    display(df.head(3))


In [ ]:

# --- Load data & split into X, y (auto-detect label) ---
import pandas as pd, numpy as np

def _pick_label(df: pd.DataFrame):
    # Prioritized common names
    for cand in ["label","y","target","TARGET","outcome"]:
        if cand in df.columns:
            return cand
    # Fallback: last column
    return df.columns[-1]

def _split_xy(df: pd.DataFrame, label_name: str):
    y = df[label_name]
    X = df.drop(columns=[label_name])
    # Coerce y to binary ints if possible
    if y.dtype.kind in "biu":
        y_bin = (y > 0).astype(int).values
    else:
        yn = y.astype(str).str.lower().map({"1":1,"true":1,"t":1,"yes":1,"y":1,"positive":1,"pos":1,
                                            "0":0,"false":0,"f":0,"no":0,"n":0,"negative":0,"neg":0})
        if yn.isna().any():
            # fallback: compare to mode
            mode = y.mode().iloc[0]
            yn = (y == mode).astype(int)
        y_bin = yn.values.astype(int)
    return X, y_bin

import pandas as pd
df_tr = pd.read_csv(CONFIG["TRAIN_PATH"])
df_va = pd.read_csv(CONFIG["VAL_PATH"])
df_te = pd.read_csv(CONFIG["TEST_PATH"])

LABEL = _pick_label(df_tr)
print("Detected label column:", LABEL)

X_tr, y_tr = _split_xy(df_tr, LABEL)
X_va, y_va = _split_xy(df_va, LABEL if LABEL in df_va.columns else _pick_label(df_va))
X_te, y_te = _split_xy(df_te, LABEL if LABEL in df_te.columns else _pick_label(df_te))

# Keep only numeric columns for the quick trainer
num_cols = X_tr.select_dtypes(include=["number"]).columns.tolist()
X_tr, X_va, X_te = X_tr[num_cols], X_va[num_cols], X_te[num_cols]

print(f"Train: X={X_tr.shape}, positives={y_tr.sum()} ({y_tr.mean():.3f})")
print(f"Valid: X={X_va.shape}, positives={y_va.sum()} ({y_va.mean():.3f})")
print(f"Test : X={X_te.shape}, positives={y_te.sum()} ({y_te.mean():.3f})")


In [ ]:

# --- Build & fit a small model (choose 'mlp' or 'logreg') ---
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

MODEL_KIND = "mlp"  # 'mlp' or 'logreg'

if MODEL_KIND == "mlp":
    est = MLPClassifier(hidden_layer_sizes=(32,16), activation="relu", solver="adam",
                        alpha=1e-4, learning_rate_init=1e-3, max_iter=200, random_state=42)
else:
    est = LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=None, random_state=42)

pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler(with_mean=True, with_std=True)),
    ("est", est),
])

pipe.fit(X_tr, y_tr)
print("Fitted base model:", type(est).__name__)


In [ ]:

# --- Isotonic calibration (fit on validation) + pick threshold s.t. recall >= 0.85 ---
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, brier_score_loss, classification_report

p_va_raw = pipe.predict_proba(X_va)[:,1]

# Fit isotonic calibrator on validation raw scores
cal = IsotonicRegression(out_of_bounds="clip")
cal.fit(p_va_raw, y_va)

def calibrate(p):
    return cal.transform(np.asarray(p))

p_va_cal = calibrate(p_va_raw)

# Choose threshold: maximum precision subject to recall >= 0.85 (if unattainable, pick smallest threshold meeting recall)
prec, rec, thr = precision_recall_curve(y_va, p_va_cal)
thr_full = np.r_[0.0, thr]  # align sizes with prec/rec
mask = rec >= 0.85
if mask.any():
    idx = np.argmax(prec[mask])  # precision-maximizing index among recall≥0.85
    thr_star = float(thr_full[mask][idx])
else:
    thr_star = float(thr_full[np.argmax(rec)])
print(f"Selected threshold (recall≥0.85 floor): {thr_star:.4f}")

# Quick validation metrics
auc_val = roc_auc_score(y_va, p_va_cal)
ap_val  = average_precision_score(y_va, p_va_cal)
brier_val = brier_score_loss(y_va, p_va_cal)
yhat_va = (p_va_cal >= thr_star).astype(int)
print("Validation AUC:", round(auc_val,4), "AP:", round(ap_val,4), "Brier:", round(brier_val,4))
print("Validation report:\n", classification_report(y_va, yhat_va, digits=3))


In [ ]:

# --- Evaluate on test ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, classification_report, confusion_matrix

p_te_raw = pipe.predict_proba(X_te)[:,1]
p_te_cal = calibrate(p_te_raw)

auc_te = roc_auc_score(y_te, p_te_cal)
ap_te  = average_precision_score(y_te, p_te_cal)
brier_te = brier_score_loss(y_te, p_te_cal)
yhat_te = (p_te_cal >= thr_star).astype(int)

tn, fp, fn, tp = confusion_matrix(y_te, yhat_te).ravel()
rec_te = tp/(tp+fn) if (tp+fn)>0 else 0.0
prec_te = tp/(tp+fp) if (tp+fp)>0 else 0.0

print(f"Test  AUC: {auc_te:.4f}  AP: {ap_te:.4f}  Brier: {brier_te:.4f}")
print("Test  report:\n", classification_report(y_te, yhat_te, digits=3))
print(f"Test  confusion: TN={tn} FP={fp} FN={fn} TP={tp}  (recall={rec_te:.3f}, precision={prec_te:.3f})")


In [ ]:

# --- Save artifacts ---
from joblib import dump
import pandas as pd, json

bundle = {
    "pipeline": pipe,
    "calibrator": cal,
    "threshold": float(thr_star),
    "features": list(X_tr.columns),
    "label": str(LABEL),
    "model_kind": "MLP" if hasattr(pipe.named_steps["est"], "hidden_layer_sizes") else "LogReg",
    "created_utc": _dt.datetime.utcnow().isoformat()+"Z",
}

dump(bundle, CONFIG["MODEL_OUT"])
print("Saved:", CONFIG["MODEL_OUT"])

# Write test predictions
pd.DataFrame({"prob_cal": p_te_cal, "y_true": y_te}).to_csv(CONFIG["PRED_OUT"], index=False)
print("Saved:", CONFIG["PRED_OUT"])

_append_event({"type":"trainer_complete",
               "model_out": CONFIG["MODEL_OUT"],
               "pred_out": CONFIG["PRED_OUT"],
               "threshold": float(thr_star)})
print("Logged to:", CONFIG["EVENT_LOG_PATH"])


In [ ]:

# --- How to use in OPS notebook ---
from joblib import load
import pandas as pd

def load_scorer(bundle_path=CONFIG["MODEL_OUT"]):
    b = load(bundle_path)
    pipe = b["pipeline"]
    cal  = b["calibrator"]
    thr  = b["threshold"]
    feats= b["features"]
    def score_proba(df: pd.DataFrame):
        X = df[feats]
        p_raw = pipe.predict_proba(X)[:,1]
        import numpy as np
        return cal.transform(np.asarray(p_raw))
    return score_proba, thr, feats, b

score_proba, thr, feats, meta = load_scorer()
print("Loaded scorer. Threshold =", thr, "Features:", len(feats))


In [ ]:

# --- Smoke test: calibrated proba on 5 test rows ---
import pandas as pd
df_test = pd.read_csv(CONFIG["TEST_PATH"])
res = score_proba(df_test.drop(columns=[LABEL]))[:5]
print("Sample calibrated probs:", [round(float(x),4) for x in res])
